
# 04 - Patient-level inference (clinical outcome, v2)

**Analysis unit: the patient.** Each of the 24 patients contributes exactly one value per
endpoint. Cells are never treated as replicates, and pooled CellPhoneDB permutation p-values
are never carried into this notebook as patient-level evidence.

## Design

24 patients: 13 Good and 11 Poor by `clinical_outcome`. Two endpoint families are tested
**independently**, each with its own Benjamini-Hochberg correction. Neither family is described
as validating the other, and neither validates the exploratory pooled-cell analysis.

| Family | Endpoint definition | Value | Expected size |
|---|---|---|---|
| `primary` | one endpoint per (`id_cp_interaction`, `sender`, `receiver`), direction preserved | the patient's CellPhoneDB interaction score for that directed cell-type pair | ~10,000-100,000 |
| `secondary` | one endpoint per `id_cp_interaction` over a **fixed** set of 36 directed pairs (6 core cell types x themselves), identical for every patient | mean score across all 36 pairs; a patient is evaluable only if **all 36** are computed | ~500-1,200 |

The secondary family never averages over "whichever pairs happen to exist" in a patient: the pair
set is fixed by construction upstream in `03_extract_patient_endpoints.ipynb`, and a patient with
any of the 36 pairs missing is treated as not evaluable for that endpoint.

## Per-endpoint procedure

1. Require >= `min_evaluable_per_group` (6) evaluable Good and >= 6 evaluable Poor patients.
2. Require >= `min_nonzero_total` (6) non-zero scores across both groups.
3. Record group sizes, non-zero counts, medians, interquartile ranges and missingness counts.
4. Two-sided Mann-Whitney U, Good versus Poor, **hybrid** p-value method (see the rationale cell below).
5. Rank-biserial correlation, sign defined as Good minus Poor.
6. Median difference, `median_good - median_poor`.
7. Stratified **patient** bootstrap 95% percentile intervals for both effect sizes
   (10,000 resamples, seed `analysis_seed`). Patients are resampled within outcome group.
   **Cells are never resampled.**
8. Benjamini-Hochberg applied **once** across all tested endpoints in that family.
9. Patient-label permutation check on the prespecified subset (BH `q < 0.10` plus the 100
   smallest raw p-values). This is a **sensitivity check, not a second discovery filter**.
10. A separate Fisher's exact test of whether endpoint *availability* differs by outcome, BH
    corrected as its own family, used to flag score comparisons that may be selection-sensitive.

## Reporting language (enforced throughout)

Permitted: "associated with clinical-outcome group", "patient-level comparison",
"exploratory pooled-cell signal", "rank-biserial correlation" with confidence interval,
"Benjamini-Hochberg adjusted q-value".

Not permitted anywhere in this notebook or its outputs: "caused", "protective",
"drives outcome", "validated", or "trend" used as a substitute for a non-significant result.
If no endpoint survives FDR, this notebook reports **zero** hits. It does not move the
threshold, promote nominal p-values, or relabel a non-significant result.

In [ ]:
import os
# Project data root. Override with the ACC_DATA_ROOT environment variable.
ACC_DATA_ROOT = os.environ.get("ACC_DATA_ROOT", os.path.expanduser("~/acc_data"))



## Configuration

One configuration cell. Everything downstream reads from `CFG` and the constants defined here;
no other cell introduces hidden state or re-reads configuration.

In [ ]:

# =====================================================================
# CONFIGURATION CELL - the single source of run parameters
# =====================================================================
from pathlib import Path
import sys

SCRIPT_DIR = Path(
    f"{ACC_DATA_ROOT}/scripts/named_scripts/"
    "cellphoneDB_scripts/clinical_outcome_v2"
)
OUTPUT_ROOT = Path(
    f"{ACC_DATA_ROOT}/outputs/cellphoneDB/clinical_outcome_v2"
)
CLINICAL_INFO_PATH = Path(f"{ACC_DATA_ROOT}/clinical_info.csv")

CONFIG_PATH = SCRIPT_DIR / "config.yaml"
PRIMARY_DIR = OUTPUT_ROOT / "primary"
SENSITIVITY_DIR = OUTPUT_ROOT / "sensitivity"
DIAG_DIR = PRIMARY_DIR / "diagnostics"
LOG_DIR = OUTPUT_ROOT / "logs"
LOG_PATH = LOG_DIR / "04_patient_level_inference.jsonl"
NOTEBOOK_ID = "04_patient_level_inference"

assert SCRIPT_DIR.is_dir(), f"script directory not found: {SCRIPT_DIR}"
assert OUTPUT_ROOT.is_dir(), f"output root not found: {OUTPUT_ROOT}"
assert CONFIG_PATH.is_file(), (
    f"config.yaml not found at {CONFIG_PATH}; it is produced by 00_audit_inputs_and_sources"
)
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))

from utils import (  # noqa: E402  shared API - imported, never reimplemented
    load_config,
    log_jsonl,
    mannwhitney_hybrid,
    rank_biserial,
    bootstrap_ci,
    permutation_p,
    bh,
    fisher_availability,
)

CFG = load_config(CONFIG_PATH)


def cfg_get(key, default=None, required=True):
    """Read a configuration key from a mapping-like or attribute-like config object."""
    if hasattr(CFG, "get") and not isinstance(CFG, (str, bytes)):
        try:
            if key in CFG:
                return CFG[key]
        except TypeError:
            pass
    if hasattr(CFG, key):
        return getattr(CFG, key)
    assert not required, f"required configuration key missing: {key}"
    return default


# --- analysis parameters (asserted against the prespecified contract) -------
MIN_EVAL_PER_GROUP = int(cfg_get("min_evaluable_per_group"))
MIN_NONZERO_TOTAL = int(cfg_get("min_nonzero_total"))
FDR_METHOD = str(cfg_get("fdr_method"))
FDR_ALPHA = float(cfg_get("fdr_alpha"))
BOOTSTRAP_RESAMPLES = int(cfg_get("bootstrap_resamples"))
PERMUTATION_RESAMPLES = int(cfg_get("permutation_resamples"))
ANALYSIS_SEED = int(cfg_get("analysis_seed"))
MW_METHOD = str(cfg_get("mannwhitney_method", "hybrid", required=False))
OUTCOME_COLUMN = str(cfg_get("outcome_column", "clinical_outcome", required=False))
OUTCOMES = list(cfg_get("outcomes", ["Good", "Poor"], required=False))

assert MIN_EVAL_PER_GROUP == 6, MIN_EVAL_PER_GROUP
assert MIN_NONZERO_TOTAL == 6, MIN_NONZERO_TOTAL
assert FDR_METHOD == "fdr_bh", FDR_METHOD
assert abs(FDR_ALPHA - 0.05) < 1e-12, FDR_ALPHA
assert BOOTSTRAP_RESAMPLES == 10000, BOOTSTRAP_RESAMPLES
assert PERMUTATION_RESAMPLES == 99999, PERMUTATION_RESAMPLES
assert ANALYSIS_SEED == 20260731, ANALYSIS_SEED
assert MW_METHOD == "hybrid", MW_METHOD
assert OUTCOME_COLUMN == "clinical_outcome", (
    "patient labels must derive only from clinical_outcome, not model_outcome"
)
assert OUTCOMES == ["Good", "Poor"], OUTCOMES

# --- derived analysis constants --------------------------------------------
CI_LEVEL = 95.0
CI_LOW_PCT = (100.0 - CI_LEVEL) / 2.0          # 2.5
CI_HIGH_PCT = 100.0 - CI_LOW_PCT               # 97.5
PERMUTATION_Q_GATE = 0.10                      # prespecified subset rule
PERMUTATION_TOP_N = 100                        # prespecified subset rule
BOOT_ENDPOINT_CHUNK = 512                      # endpoints per bootstrap chunk
PERM_CHUNK = 2500                              # permutations per chunk
AUDIT_SAMPLE_N = 400                           # endpoints cross-checked against utils

# --- expected design (recorded, and asserted where structural) -------------
EXPECTED_N_PATIENTS = 24
EXPECTED_N_GOOD = 13
EXPECTED_N_POOR = 11
EXPECTED_SECONDARY_PAIRS = 36
EXPECTED_SECONDARY_EVALUABLE = {"total": 20, "good": 11, "poor": 9}  # expectation, not a gate

# --- input tables produced by 03_extract_patient_endpoints -----------------
SCORES_LONG_PATH = PRIMARY_DIR / "patient_endpoint_scores_long.parquet"
AVAILABILITY_PATH = PRIMARY_DIR / "patient_endpoint_availability.parquet"
ENDPOINT_META_PATH = PRIMARY_DIR / "endpoint_metadata.parquet"
# The secondary fixed-pair-set table name is resolved defensively below; an explicit
# config key wins if 03_ recorded one.
SECONDARY_TABLE_CONFIG = cfg_get("secondary_fixed_pair_table", None, required=False)
SECONDARY_NAME_CANDIDATES = [
    "patient_endpoint_scores_secondary_fixed_pairs.parquet",
    "patient_interaction_scores_fixed_pairs.parquet",
    "patient_endpoint_scores_secondary.parquet",
    "secondary_fixed_pair_scores.parquet",
    "patient_fixed_pair_interaction_scores.parquet",
]
SECONDARY_GLOBS = ["*secondary*.parquet", "*fixed_pair*.parquet", "*fixed*pair*.parquet"]

# --- write guard: nothing outside the v2 output tree is ever touched -------
FORBIDDEN_PATH_TOKENS = (
    "cpdb_results_",
    "t1_cpdb_results",
    "t2_cpdb_results",
    "t3_cpdb_results",
    "cellphoneDB_copy_backup",
)
for d in (PRIMARY_DIR, SENSITIVITY_DIR, DIAG_DIR, LOG_DIR):
    d.mkdir(parents=True, exist_ok=True)

In [ ]:

# =====================================================================
# Imports and environment record (no configuration is defined here)
# =====================================================================
import datetime as dt
import hashlib
import json
import platform
import warnings

import numpy as np
import pandas as pd
import scipy
import statsmodels
from scipy.stats import norm, rankdata

from IPython.display import display

ENVIRONMENT = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
    "statsmodels": statsmodels.__version__,
}
assert tuple(int(x) for x in scipy.__version__.split(".")[:2]) >= (1, 15), (
    "scipy >= 1.15 is required: rankdata(axis=...) and mannwhitneyu(method='exact') "
    f"semantics are assumed; found {scipy.__version__}"
)

RUN_STARTED = dt.datetime.now().isoformat(timespec="seconds")
CONFIG_SHA256 = hashlib.sha256(CONFIG_PATH.read_bytes()).hexdigest()


def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def guarded_path(path):
    """Assert a write target is inside the v2 output tree and touches no legacy result."""
    p = Path(path).resolve()
    assert OUTPUT_ROOT.resolve() in p.parents, f"refusing to write outside output root: {p}"
    for token in FORBIDDEN_PATH_TOKENS:
        assert token not in str(p), f"refusing to write into a protected location: {p}"
    p.parent.mkdir(parents=True, exist_ok=True)
    return p


WRITTEN_FILES = {}


def write_table(df, path, index=False):
    p = guarded_path(path)
    df.to_csv(p, index=index)
    WRITTEN_FILES[str(p)] = {"rows": int(len(df)), "sha256": _sha256(p)}
    return p


def _try_calls(fn, *argsets, **kwargs):
    """Call `fn` with the first argument tuple its signature accepts.

    The shared `utils` API is authoritative but is authored concurrently; this keeps the
    notebook robust to argument-order differences without reimplementing any statistic.
    """
    errors = []
    for args in argsets:
        try:
            return fn(*args, **kwargs)
        except TypeError as exc:  # signature mismatch only
            errors.append(f"{args!r}: {exc}")
    raise TypeError(f"could not call {getattr(fn, '__name__', fn)}; tried: {errors}")


def LOG(event, **fields):
    record = {
        "notebook": NOTEBOOK_ID,
        "event": event,
        "timestamp": dt.datetime.now().isoformat(timespec="seconds"),
        "config_sha256": CONFIG_SHA256,
        **fields,
    }
    guarded_path(LOG_PATH)
    try:
        _try_calls(log_jsonl, (LOG_PATH, record), (record, LOG_PATH))
    except (TypeError, OSError):
        with open(LOG_PATH, "a") as fh:
            fh.write(json.dumps(record, default=str) + "\n")
    return record


LOG("start", environment=ENVIRONMENT, run_started=RUN_STARTED)


## Why the Mann-Whitney p-value method is hybrid, and why it decides the whole analysis

At 13 Good versus 11 Poor patients the most extreme result the design can produce is perfect
separation: all 13 Good values above all 11 Poor values (or the mirror image). That single
configuration is the entire lower bound of the p-value scale, and its value depends on which
method computes it:

| method | two-sided p for perfect separation at 13 v 11 |
|---|---|
| `method='asymptotic'` (normal approximation, continuity corrected) | 3.897139e-05 |
| `method='exact'` (permutation null of U) | 8.012358e-07 |

Under Benjamini-Hochberg across a family of roughly 20,000 endpoints, the smallest attainable
q-value for a single most-extreme endpoint is approximately `m * p`. With the asymptotic value
that is `20000 * 3.897e-05 = 0.78`; with the exact value it is `20000 * 8.012e-07 = 0.016`.
**Using the asymptotic p-value everywhere would make the primary analysis return zero hits by
arithmetic rather than by biology** - no endpoint could reach `q < 0.05` no matter how cleanly
it separated the outcome groups. That is a property of the p-value method, not of the data, and
it must not be mistaken for a biological finding.

The exact method is therefore used wherever it is valid. It is **not** valid under ties: the
exact null distribution of U assumes a tie-free ranking, and `scipy.stats.mannwhitneyu` with
`method='exact'` ignores ties rather than correcting for them. Ties are expected here, because
CellPhoneDB interaction scores contain many exact zeros, so an endpoint with several zero-valued
patients has a genuinely tied ranking.

The hybrid rule is therefore:

* endpoint is **tie-free** across its evaluable patients -> `method='exact'`;
* endpoint has **any tie** -> `method='asymptotic'` with the standard tie correction to the
  variance of U.

`p_method` is recorded per endpoint so that every reported q-value can be traced to the method
that produced it. Two consequences are stated openly rather than hidden:

1. The two methods are on slightly different scales, so a tied endpoint is at an inherent
   disadvantage in the BH ranking relative to a tie-free endpoint with the same effect size.
   This is a real, unavoidable feature of rank tests on data with point masses at zero.
2. Any endpoint whose p-value floor exceeds `alpha / m` cannot be rejected at rank 1 regardless
   of its effect size. That is quantified in the attainability diagnostic near the end of this
   notebook and is deliberately **not** acted upon in this run.


## Adapters over the shared `utils` API

Every statistic in this notebook is defined by `utils`. Nothing is reimplemented. Two things are
added on top:

* **normalisers** - small functions that accept whatever shape a `utils` function returns
  (tuple, dict or result object) and hand back plain numbers;
* **vectorised sweeps** - array implementations of the *same* statistic applied to all endpoints
  at once, because a per-endpoint Python loop over 10^4-10^5 endpoints x 10^4 bootstrap resamples
  cannot finish. Each sweep is asserted, on a random audit sample of real endpoints, to agree
  with the corresponding `utils` function to 1e-9. If an assertion fires, the sweep and the
  shared API have diverged and the run stops.

In [ ]:

# =====================================================================
# Normalisers for the shared utils API
# =====================================================================
def _first_number(obj, names):
    for nm in names:
        if isinstance(obj, dict) and nm in obj:
            return obj[nm]
        if hasattr(obj, nm):
            return getattr(obj, nm)
    return None


def as_mw_result(res):
    """-> (u_statistic, p_value, p_method) from mannwhitney_hybrid output."""
    u = _first_number(res, ("u", "U", "statistic", "mannwhitney_u", "u_statistic"))
    p = _first_number(res, ("p", "pvalue", "p_value"))
    m = _first_number(res, ("method", "p_method", "mw_method"))
    if u is None and isinstance(res, (tuple, list)):
        if len(res) == 3:
            u, p, m = res
        elif len(res) == 2:
            u, p = res
    assert u is not None and p is not None, f"unrecognised mannwhitney_hybrid return: {res!r}"
    method = str(m).lower() if m is not None else ""
    if "exact" in method:
        method = "exact"
    elif "asym" in method or "normal" in method:
        method = "asymptotic"
    else:
        method = method or "unknown"
    return float(u), float(p), method


def as_scalar(res, names=("estimate", "value", "statistic", "r", "correlation")):
    v = _first_number(res, names)
    if v is None:
        if isinstance(res, (tuple, list)) and len(res) >= 1:
            v = res[0]
        else:
            v = res
    return float(v)


def as_interval(res):
    """-> (low, high) from a CI-shaped return."""
    lo = _first_number(res, ("low", "lower", "ci_low", "lo"))
    hi = _first_number(res, ("high", "upper", "ci_high", "hi"))
    if lo is None or hi is None:
        arr = np.asarray(res, dtype=float).ravel()
        assert arr.size >= 2, f"unrecognised interval return: {res!r}"
        lo, hi = arr[-2], arr[-1]
    return float(lo), float(hi)


def as_qvalues(res, n):
    """-> q-value array from a bh() return (array, or (reject, q), or dict)."""
    q = _first_number(res, ("q", "qvalues", "q_values", "pvals_corrected", "fdr"))
    if q is None:
        if isinstance(res, (tuple, list)):
            for item in res:
                arr = np.asarray(item)
                if arr.dtype.kind == "f" and arr.size == n:
                    q = arr
                    break
        else:
            q = res
    q = np.asarray(q, dtype=float).ravel()
    assert q.size == n, f"bh() returned {q.size} q-values for {n} p-values"
    return q


def as_or_p(res):
    """-> (odds_ratio, p_value) from fisher_availability output."""
    orr = _first_number(res, ("odds_ratio", "oddsratio", "or", "statistic"))
    p = _first_number(res, ("p", "pvalue", "p_value"))
    if orr is None or p is None:
        arr = list(res) if isinstance(res, (tuple, list)) else [res]
        if len(arr) >= 2:
            orr, p = arr[0], arr[1]
    assert orr is not None and p is not None, f"unrecognised fisher_availability return: {res!r}"
    return float(orr), float(p)


BACKENDS = {}


def bh_qvalues(pvals):
    """Benjamini-Hochberg via the shared API, with a recorded statsmodels fallback."""
    p = np.asarray(pvals, dtype=float)
    assert p.size > 0, "BH called on an empty family"
    assert np.all(np.isfinite(p)) and np.all((p >= 0) & (p <= 1)), "non-finite p-value in BH input"
    try:
        res = _try_calls(bh, (p,), (p, FDR_ALPHA), (list(p),))
        q = as_qvalues(res, p.size)
        BACKENDS["bh"] = "utils.bh"
    except (TypeError, AssertionError) as exc:
        from statsmodels.stats.multitest import multipletests

        q = multipletests(p, alpha=FDR_ALPHA, method="fdr_bh")[1]
        BACKENDS["bh"] = f"statsmodels.multipletests (utils.bh unusable: {exc})"
    q = np.clip(np.asarray(q, dtype=float), 0.0, 1.0)
    # structural properties of BH that must hold whichever backend produced them
    order = np.argsort(p, kind="stable")
    assert np.all(np.diff(q[order]) >= -1e-12), "q-values are not monotone in p"
    assert np.all(q + 1e-12 >= p), "a q-value fell below its own p-value"
    return q


def fisher_2x2(a_good, n_good, a_poor, n_poor):
    """Fisher exact odds ratio and p for an availability 2x2 table via the shared API."""
    table = [[int(a_good), int(n_good - a_good)], [int(a_poor), int(n_poor - a_poor)]]
    try:
        res = _try_calls(
            fisher_availability,
            (np.array([int(a_good), int(n_good)], dtype=int),
             np.array([int(a_poor), int(n_poor)], dtype=int)),
            (table,),
            (np.asarray(table),),
            (table[0][0], table[0][1], table[1][0], table[1][1]),
            (a_good, n_good, a_poor, n_poor),
        )
        BACKENDS.setdefault("fisher", "utils.fisher_availability")
        return as_or_p(res)
    except (TypeError, AssertionError, ValueError) as exc:
        from scipy.stats import fisher_exact

        BACKENDS["fisher"] = f"scipy.fisher_exact (utils.fisher_availability unusable: {exc})"
        r = fisher_exact(table)
        return float(r[0]), float(r[1])


## Vectorised statistical kernel

Three sweeps, all built on the same rank matrix so that the observed test, the permutation null
and the bootstrap cannot drift apart.

**Rank matrix.** Unavailable patient-endpoint cells are filled with `+inf` before ranking, so
they always sort last and never disturb the ranks of the evaluable values, then they are masked
out of every sum. This ranks *all* endpoints in a single `scipy.stats.rankdata(..., axis=1)`
call regardless of how missingness is patterned.

**Mann-Whitney sweep.** With `R` the rank matrix, `A` the availability mask and `g` the Good
indicator over the 24 patients:

```
n_good = A @ g                     n_poor = A @ (1 - g)
U_good = (R * A) @ g - n_good (n_good + 1) / 2
```

one matrix-vector product per family. The tie-corrected variance uses a per-endpoint tie term
computed from run lengths in the row-sorted score matrix by a `bincount` over group ids - no
Python loop. The asymptotic p-value reproduces
`scipy.stats.mannwhitneyu(..., method='asymptotic')` to 0 ulp (verified in the self-test cell
below). Per-endpoint `scipy`/`utils` calls are reserved for the tie-free endpoints that take the
**exact** method, and even those are memoised on `(n_good, n_poor, U_good)`, because for tie-free
data the exact p-value is a function of exactly those three numbers.

**Bootstrap sweep.** Endpoints are grouped by identical availability pattern, so within a group
the evaluable Good set and Poor set are the same patients and one set of resample indices serves
every endpoint in it. Index arrays are drawn **once** per distinct group size from a single
`np.random.default_rng(analysis_seed)`, in sorted size order, so the stream is deterministic and
independent of the order endpoints happen to be processed in. Then:

* *rank-biserial:* pre-compute `H[e, i, j] = 1(g_i > p_j) + 0.5 * 1(g_i == p_j)` once per
  endpoint and flatten it. For resample `b`, `U_b = sum_ij c_g[b, i] c_p[b, j] H[e, i, j]`,
  which is the inner product of `H[e]` with the outer product `c_g[b] (x) c_p[b]`. Stacking those
  outer products into a matrix `K` turns all 10,000 resamples for a whole chunk of endpoints into
  a single `(k x n_good n_poor) @ (n_good n_poor x 10000)` BLAS matmul. No resample is ever
  materialised as a value array.
* *median difference:* resampling positions in the **sorted** value vector makes the resampled
  median an order statistic of the sorted draw indices. The two index positions that define the
  median are therefore identical for every endpoint in the block and are computed once, so the
  bootstrap median is a pure gather: `0.5 * (Vsorted[:, lo_b] + Vsorted[:, hi_b])`, with no
  per-resample sort at all.

**Permutation sweep.** Permuting patient labels does not change which patients are evaluable and
does not change any score, so the rank matrix and the tie terms are computed once and reused for
all 99,999 permutations. Each permutation is then a column of a 24 x B indicator matrix and the
whole null is two matmuls, chunked over permutations.

In [ ]:

# =====================================================================
# Vectorised kernel (identical statistics to utils, applied family-wide)
# =====================================================================
def rank_matrix_and_tie_terms(V, A):
    """Average ranks within each endpoint's evaluable set, plus sum(t^3 - t) per endpoint."""
    V = np.asarray(V, dtype=float)
    A = np.asarray(A, dtype=bool)
    assert V.shape == A.shape
    assert not np.any(np.isnan(V[A])), "NaN score at an available patient-endpoint cell"
    filled = np.where(A, V, np.inf)
    R = rankdata(filled, axis=1, method="average")

    order = np.argsort(filled, axis=1, kind="stable")
    S = np.take_along_axis(filled, order, axis=1)
    fin = np.isfinite(S)
    neq = np.ones(S.shape, dtype=bool)
    neq[:, 1:] = S[:, 1:] != S[:, :-1]
    gid = np.cumsum(neq, axis=1) - 1
    k, n = S.shape
    flat = (np.repeat(np.arange(k), n) * n + gid.ravel())
    keep = fin.ravel()
    counts = np.bincount(flat[keep], minlength=k * n)
    t_elem = np.where(keep, counts[flat], 0).reshape(k, n).astype(np.float64)
    tie_term = (t_elem ** 2).sum(axis=1) - fin.sum(axis=1)   # == sum(t^3 - t)
    return R, tie_term


def mw_asymptotic_from_u(U1, n1, n2, tie_term):
    """Two-sided tie-corrected normal-approximation p, matching scipy's convention."""
    n1 = np.asarray(n1, dtype=float)
    n2 = np.asarray(n2, dtype=float)
    U1 = np.asarray(U1, dtype=float)
    U2 = n1 * n2 - U1
    n = n1 + n2
    with np.errstate(invalid="ignore", divide="ignore"):
        sd = np.sqrt(n1 * n2 / 12.0 * ((n + 1.0) - tie_term / (n * (n - 1.0))))
        z = (np.maximum(U1, U2) - n1 * n2 / 2.0 - 0.5) / sd
        p = np.clip(2.0 * norm.sf(z), 0.0, 1.0)
    return p, z, sd


def mw_sweep(V, A, good_vec, poor_vec):
    """U, group sizes, tie terms and asymptotic p for every endpoint at once."""
    R, tie_term = rank_matrix_and_tie_terms(V, A)
    Af = A.astype(np.float64)
    n1 = Af @ good_vec
    n2 = Af @ poor_vec
    U1 = (R * Af) @ good_vec - n1 * (n1 + 1.0) / 2.0
    p_asym, z, sd = mw_asymptotic_from_u(U1, n1, n2, tie_term)
    return {
        "R": R, "tie_term": tie_term, "n_good": n1.astype(int), "n_poor": n2.astype(int),
        "u_good": U1, "p_asymptotic": p_asym, "z": z, "sd": sd,
    }


def rank_biserial_from_u(U1, n1, n2):
    """Rank-biserial correlation, sign = Good minus Poor."""
    with np.errstate(invalid="ignore", divide="ignore"):
        return 2.0 * np.asarray(U1, float) / (np.asarray(n1, float) * np.asarray(n2, float)) - 1.0


def resample_counts(idx, n):
    """Multiplicity of each original position in every bootstrap resample."""
    b = idx.shape[0]
    flat = (np.arange(b)[:, None] * n + idx).ravel()
    return np.bincount(flat, minlength=b * n).reshape(b, n).astype(np.float64)


def draw_bootstrap_indices(sizes, n_resamples, seed):
    """Draw all resample index arrays ONCE, in sorted size order, from a single RNG."""
    rng = np.random.default_rng(seed)
    out = {}
    for s in sorted(int(x) for x in sizes):
        idx = rng.integers(0, s, size=(n_resamples, s))
        srt = np.sort(idx, axis=1)
        out[s] = {
            "idx": idx,
            "counts": resample_counts(idx, s),
            "med_lo": srt[:, (s - 1) // 2],
            "med_hi": srt[:, s // 2],
        }
    return out


def permutation_label_matrix(rng, n_patients, n_good, n_perm):
    """(n_patients x n_perm) 0/1 Good-indicator matrix, each column exactly n_good ones."""
    order = np.argsort(rng.random((n_perm, n_patients)), axis=1)
    L = np.zeros((n_perm, n_patients), dtype=np.float64)
    np.put_along_axis(L, order[:, :n_good], 1.0, axis=1)
    return L.T

In [ ]:

# =====================================================================
# Self-test: the vectorised kernel against scipy and against utils
# =====================================================================
def _selftest_kernel():
    rng = np.random.default_rng(11)
    n_pat, k = 24, 300
    V = rng.random((k, n_pat))
    V[V < 0.45] = 0.0                                  # CPDB-like point mass at zero
    V[rng.random((k, n_pat)) < 0.18] = np.nan          # structural missingness
    A = np.isfinite(V)
    g = np.zeros(n_pat); g[:13] = 1.0
    p_ = 1.0 - g
    sw = mw_sweep(V, A, g, p_)

    from scipy.stats import mannwhitneyu
    max_du = max_dp = 0.0
    n_checked = 0
    for e in range(k):
        gv = V[e][A[e] & (g > 0)]
        pv = V[e][A[e] & (p_ > 0)]
        if len(gv) < 2 or len(pv) < 2:
            continue
        r = mannwhitneyu(gv, pv, alternative="two-sided", method="asymptotic")
        max_du = max(max_du, abs(r.statistic - sw["u_good"][e]))
        max_dp = max(max_dp, abs(r.pvalue - sw["p_asymptotic"][e]))
        assert len(gv) == sw["n_good"][e] and len(pv) == sw["n_poor"][e]
        n_checked += 1
    assert n_checked > 100 and max_du == 0.0 and max_dp < 1e-12, (n_checked, max_du, max_dp)

    # tie-free rows must be flagged tie-free, and exact p must depend only on (n1, n2, U)
    Vt = rng.permuted(np.sort(rng.random((40, n_pat)), axis=1), axis=1)
    At = np.ones_like(Vt, dtype=bool)
    swt = mw_sweep(Vt, At, g, p_)
    assert np.all(swt["tie_term"] == 0), "tie-free data reported ties"
    seen = {}
    for e in range(Vt.shape[0]):
        r = mannwhitneyu(Vt[e, :13], Vt[e, 13:], alternative="two-sided", method="exact")
        key = (13, 11, float(swt["u_good"][e]))
        assert seen.setdefault(key, r.pvalue) == r.pvalue, "exact p is not a function of the key"
    assert abs(mannwhitneyu(np.arange(13) + 100.0, np.arange(11.0),
                            alternative="two-sided", method="exact").pvalue
               - 8.012358261382357e-07) < 1e-18
    assert abs(mannwhitneyu(np.arange(13) + 100.0, np.arange(11.0),
                            alternative="two-sided", method="asymptotic").pvalue
               - 3.897138871111154e-05) < 1e-15

    # bootstrap identities
    b, n1, n2 = 500, 13, 11
    draws = draw_bootstrap_indices([n1, n2], b, ANALYSIS_SEED)
    Vg = rng.random((6, n1)); Vg[Vg < 0.4] = 0.0
    Vp = rng.random((6, n2)); Vp[Vp < 0.4] = 0.0
    cg, cp = draws[n1]["counts"], draws[n2]["counts"]
    K = (cg[:, :, None] * cp[:, None, :]).reshape(b, n1 * n2).T
    H = ((Vg[:, :, None] > Vp[:, None, :]).astype(float)
         + 0.5 * (Vg[:, :, None] == Vp[:, None, :])).reshape(6, n1 * n2)
    Ub = H @ K
    gi, pi = draws[n1]["idx"], draws[n2]["idx"]
    for e in range(3):
        for bb in (0, 7, 123):
            gs, ps = Vg[e][gi[bb]], Vp[e][pi[bb]]
            hb = (gs[:, None] > ps[None, :]).astype(float) + 0.5 * (gs[:, None] == ps[None, :])
            assert abs(Ub[e, bb] - hb.sum()) < 1e-9
    Vgs, Vps = np.sort(Vg, axis=1), np.sort(Vp, axis=1)
    mg = 0.5 * (Vgs[:, draws[n1]["med_lo"]] + Vgs[:, draws[n1]["med_hi"]])
    mp = 0.5 * (Vps[:, draws[n2]["med_lo"]] + Vps[:, draws[n2]["med_hi"]])
    for e in range(3):
        for bb in (0, 7, 123):
            assert abs(mg[e, bb] - np.median(Vgs[e][gi[bb]])) < 1e-12
            assert abs(mp[e, bb] - np.median(Vps[e][pi[bb]])) < 1e-12

    # permutation identity: fixed ranks, permuted labels
    prng = np.random.default_rng(ANALYSIS_SEED)
    L = permutation_label_matrix(prng, n_pat, 13, 25)
    assert np.all(L.sum(axis=0) == 13)
    Af = A.astype(float)
    n1p = Af @ L
    U1p = (sw["R"] * Af) @ L - n1p * (n1p + 1.0) / 2.0
    for bb in (0, 4, 17):
        sel = L[:, bb].astype(bool)
        for e in (0, 5, 31):
            gv, pv = V[e][A[e] & sel], V[e][A[e] & ~sel]
            if len(gv) < 2 or len(pv) < 2:
                continue
            assert abs(mannwhitneyu(gv, pv, alternative="two-sided",
                                    method="asymptotic").statistic - U1p[e, bb]) < 1e-9
    return {"mw_endpoints_checked": n_checked, "max_u_error": max_du, "max_p_error": max_dp}


SELFTEST = _selftest_kernel()
LOG("kernel_selftest_passed", **{k: float(v) for k, v in SELFTEST.items()})
display(pd.Series(SELFTEST, name="kernel self-test"))


## Patient labels

Labels come from `clinical_info.csv` via the `clinical_outcome` column only. No patient list is
hardcoded anywhere: the analysed cohort is the intersection of patients present in the endpoint
tables with patients carrying a clinical outcome. `Model outcome` is never read.

In [ ]:

# =====================================================================
# Patient labels (derived, never hardcoded)
# =====================================================================
clinical = pd.read_csv(CLINICAL_INFO_PATH, encoding="utf-8-sig")
clinical.columns = [c.strip() for c in clinical.columns]
assert "Sample ID" in clinical.columns and "Clinical outcome" in clinical.columns, clinical.columns
assert clinical["Sample ID"].is_unique

CLINICAL = (
    clinical.rename(columns={"Sample ID": "patient", "Clinical outcome": "clinical_outcome"})
    .assign(patient=lambda d: d["patient"].astype(str).str.strip(),
            clinical_outcome=lambda d: d["clinical_outcome"].astype(str).str.strip())
)
assert set(CLINICAL["clinical_outcome"]) <= set(OUTCOMES), set(CLINICAL["clinical_outcome"])
OUTCOME_MAP = dict(zip(CLINICAL["patient"], CLINICAL["clinical_outcome"]))


def build_label_vectors(patients):
    """-> (good_vec, poor_vec) float indicator arrays aligned with `patients`."""
    labels = np.array([OUTCOME_MAP[p] for p in patients])
    good_vec = (labels == "Good").astype(np.float64)
    poor_vec = (labels == "Poor").astype(np.float64)
    assert np.all(good_vec + poor_vec == 1.0), "a patient carries neither Good nor Poor"
    return good_vec, poor_vec, labels


## Inputs from `03_extract_patient_endpoints.ipynb`

Column names are resolved from the actual file rather than assumed, and every resolution is
recorded in the run log. Structural missingness is preserved: a patient-endpoint cell that was
never computed stays `NaN` and is never coerced to zero, while a computed score of zero stays
zero. The availability table is treated as authoritative and is cross-checked against the score
table; a disagreement stops the run.

**Environment note.** The `cpdb` environment as inventoried for this run has neither `pyarrow`
nor `fastparquet` installed, so `pandas.read_parquet` is unavailable in it. The loader below
detects this and falls back to a same-stem `.csv`, `.csv.gz` or `.tsv` sibling of each declared
Parquet input. If neither form exists the notebook stops with an actionable message. Installing
`pyarrow` into `cpdb` and keeping Parquet as the authoritative format is the preferred fix; the
fallback exists so that a missing optional dependency cannot silently change the analysis.

In [ ]:

# =====================================================================
# Schema-tolerant loaders
# =====================================================================
PATIENT_ALIASES = ["patient", "patient_id", "sample", "sample_id", "batch", "Sample ID"]
SENDER_ALIASES = ["sender", "cell_type_a", "source", "celltype_a", "cell_a"]
RECEIVER_ALIASES = ["receiver", "cell_type_b", "target", "celltype_b", "cell_b"]
SCORE_ALIASES = ["score", "interaction_score", "value", "mean_score", "cpdb_score",
                 "secondary_score"]
INTERACTION_ALIASES = ["id_cp_interaction", "id_cp_interactions", "interaction_id"]
NAME_ALIASES = ["interaction_name", "interacting_pair", "interaction", "partner_pair"]
AVAILABLE_ALIASES = ["available", "is_available", "evaluable", "evaluated", "is_evaluable"]
STATUS_ALIASES = ["status", "availability", "availability_status", "classification"]


def resolve_column(df, aliases, role, required=True):
    lower = {str(c).strip().lower(): c for c in df.columns}
    for a in aliases:
        if a.lower() in lower:
            return lower[a.lower()]
    assert not required, (
        f"could not resolve the {role} column; tried {aliases}; available: {list(df.columns)}"
    )
    return None


def _parquet_engine():
    for mod in ("pyarrow", "fastparquet"):
        try:
            __import__(mod)
            return mod
        except ImportError:
            continue
    return None


PARQUET_ENGINE = _parquet_engine()
TABLE_SUFFIXES = [".parquet", ".csv", ".csv.gz", ".tsv", ".feather"]


def table_candidates(base):
    base = Path(base)
    stem = base.name[: -len(base.suffix)] if base.suffix else base.name
    out = [base] + [base.with_name(stem + s) for s in TABLE_SUFFIXES]
    return list(dict.fromkeys(out))


def read_table_any(base, role):
    """Read a table produced by 03_, tolerating the absence of a Parquet engine."""
    tried = []
    for p in table_candidates(base):
        tried.append(str(p))
        if not p.is_file():
            continue
        if p.suffix in (".parquet", ".feather"):
            if PARQUET_ENGINE is None:
                continue
            return (pd.read_parquet(p) if p.suffix == ".parquet" else pd.read_feather(p)), p
        if p.suffix == ".tsv":
            return pd.read_csv(p, sep="\t"), p
        if p.name.endswith(".csv") or p.name.endswith(".csv.gz"):
            return pd.read_csv(p), p
    engine_note = (
        "" if PARQUET_ENGINE else
        " No Parquet engine is installed in this environment (neither pyarrow nor fastparquet), "
        "so .parquet inputs cannot be read here; install pyarrow into the cpdb environment or "
        "have 03_extract_patient_endpoints also emit a .csv sibling.")
    raise AssertionError(f"could not read the {role} table. Tried: {tried}.{engine_note}")


def resolve_secondary_path():
    if SECONDARY_TABLE_CONFIG:
        p = Path(SECONDARY_TABLE_CONFIG)
        if not p.is_absolute():
            p = PRIMARY_DIR / p
        for c in table_candidates(p):
            if c.is_file():
                return c
        raise AssertionError(f"configured secondary table not found: {p}")
    for nm in SECONDARY_NAME_CANDIDATES:
        for c in table_candidates(PRIMARY_DIR / nm):
            if c.is_file():
                return c
    hits = []
    for pat in SECONDARY_GLOBS:
        for suf in TABLE_SUFFIXES:
            hits += [p for p in sorted(PRIMARY_DIR.glob(pat.replace(".parquet", suf)))
                     if p.is_file()]
    hits = [p for p in dict.fromkeys(hits) if "results" not in p.name]
    assert len(hits) >= 1, (
        "no secondary fixed-pair-set table found in "
        f"{PRIMARY_DIR}; set `secondary_fixed_pair_table` in config.yaml")
    stems = {p.name.split(".")[0] for p in hits}
    assert len(stems) == 1, f"ambiguous secondary table candidates: {[p.name for p in hits]}"
    return hits[0]


def long_to_matrix(df, endpoint_cols, patient_col, value_col, patients):
    """Pivot a long table to (endpoints x patients), NaN where not computed."""
    wide = df.pivot_table(index=endpoint_cols, columns=patient_col, values=value_col,
                          aggfunc="first", dropna=False)
    dup = df.duplicated(subset=endpoint_cols + [patient_col]).sum()
    assert dup == 0, f"{dup} duplicate patient-endpoint keys in the input table"
    wide = wide.reindex(columns=patients)
    keys = wide.index.to_frame(index=False)
    return keys, wide.to_numpy(dtype=float)


def availability_to_matrix(av, endpoint_cols, patient_col, patients, keys):
    """Build the boolean availability matrix aligned to `keys` row order."""
    keys = keys[endpoint_cols]
    acol = resolve_column(av, AVAILABLE_ALIASES, "availability flag", required=False)
    if acol is None:
        scol = resolve_column(av, STATUS_ALIASES, "availability status", required=True)
        av = av.assign(_avail=av[scol].astype(str).str.lower().isin(
            {"evaluated", "available", "evaluable", "computed", "ok", "true"}))
        acol = "_avail"
    wide = av.pivot_table(index=endpoint_cols, columns=patient_col, values=acol,
                          aggfunc="first", dropna=False).reindex(columns=patients)
    wide = wide.reindex(pd.MultiIndex.from_frame(keys) if len(endpoint_cols) > 1
                        else pd.Index(keys[endpoint_cols[0]], name=endpoint_cols[0]))
    return np.nan_to_num(wide.to_numpy(dtype=float), nan=0.0).astype(bool)

In [ ]:

# =====================================================================
# Load the primary family
# =====================================================================
scores_long, SCORES_LONG_PATH = read_table_any(SCORES_LONG_PATH, "patient endpoint scores")
availability_long, AVAILABILITY_PATH = read_table_any(AVAILABILITY_PATH, "endpoint availability")
endpoint_meta, ENDPOINT_META_PATH = read_table_any(ENDPOINT_META_PATH, "endpoint metadata")
LOG("input_formats", parquet_engine=PARQUET_ENGINE,
    scores=str(SCORES_LONG_PATH), availability=str(AVAILABILITY_PATH),
    metadata=str(ENDPOINT_META_PATH))

SCH = {
    "patient": resolve_column(scores_long, PATIENT_ALIASES, "patient"),
    "interaction": resolve_column(scores_long, INTERACTION_ALIASES, "id_cp_interaction"),
    "sender": resolve_column(scores_long, SENDER_ALIASES, "sender"),
    "receiver": resolve_column(scores_long, RECEIVER_ALIASES, "receiver"),
    "score": resolve_column(scores_long, SCORE_ALIASES, "score"),
}
LOG("primary_schema_resolved", **SCH,
    scores_long_sha256=_sha256(SCORES_LONG_PATH), rows=int(len(scores_long)))

scores_long[SCH["patient"]] = scores_long[SCH["patient"]].astype(str).str.strip()
PATIENTS = sorted(set(scores_long[SCH["patient"]]) & set(OUTCOME_MAP))
assert len(PATIENTS) == EXPECTED_N_PATIENTS, (
    f"expected {EXPECTED_N_PATIENTS} labelled patients in the endpoint table, got {len(PATIENTS)}"
)
GOOD_VEC, POOR_VEC, PATIENT_LABELS = build_label_vectors(PATIENTS)
N_GOOD_TOTAL, N_POOR_TOTAL = int(GOOD_VEC.sum()), int(POOR_VEC.sum())
assert (N_GOOD_TOTAL, N_POOR_TOTAL) == (EXPECTED_N_GOOD, EXPECTED_N_POOR), (
    N_GOOD_TOTAL, N_POOR_TOTAL)

PRIMARY_KEY_COLS = [SCH["interaction"], SCH["sender"], SCH["receiver"]]
primary_keys, V_PRIMARY = long_to_matrix(
    scores_long, PRIMARY_KEY_COLS, SCH["patient"], SCH["score"], PATIENTS)

av_patient_col = resolve_column(availability_long, PATIENT_ALIASES, "patient")
availability_long[av_patient_col] = availability_long[av_patient_col].astype(str).str.strip()
av_key_cols = [resolve_column(availability_long, INTERACTION_ALIASES, "id_cp_interaction"),
               resolve_column(availability_long, SENDER_ALIASES, "sender"),
               resolve_column(availability_long, RECEIVER_ALIASES, "receiver")]
A_DECLARED = availability_to_matrix(
    availability_long, av_key_cols, av_patient_col, PATIENTS,
    primary_keys.rename(columns=dict(zip(PRIMARY_KEY_COLS, av_key_cols))))

A_OBSERVED = np.isfinite(V_PRIMARY)
mismatch = int((A_DECLARED != A_OBSERVED).sum())
assert mismatch == 0, (
    f"{mismatch} patient-endpoint cells disagree between the availability table and the score "
    "table; 03_extract_patient_endpoints must be reconciled before inference"
)
A_PRIMARY = A_OBSERVED
V_PRIMARY = np.where(A_PRIMARY, V_PRIMARY, np.nan)
assert np.all(np.isfinite(V_PRIMARY[A_PRIMARY])), "non-finite score at an available cell"

primary_keys = primary_keys.rename(columns={
    SCH["interaction"]: "id_cp_interaction", SCH["sender"]: "sender",
    SCH["receiver"]: "receiver"})

name_col = resolve_column(endpoint_meta, NAME_ALIASES, "interaction name", required=False)
meta_id = resolve_column(endpoint_meta, INTERACTION_ALIASES, "id_cp_interaction")
if name_col is not None:
    NAME_MAP = (endpoint_meta[[meta_id, name_col]].drop_duplicates(subset=[meta_id])
                .set_index(meta_id)[name_col].astype(str).to_dict())
else:
    NAME_MAP = {}
primary_keys["interaction_name"] = primary_keys["id_cp_interaction"].map(NAME_MAP).fillna("")

LOG("primary_loaded", n_endpoints=int(len(primary_keys)), n_patients=len(PATIENTS),
    n_good=N_GOOD_TOTAL, n_poor=N_POOR_TOTAL,
    availability_agreement=True)
display(pd.Series({"primary endpoints": len(primary_keys), "patients": len(PATIENTS),
                   "Good": N_GOOD_TOTAL, "Poor": N_POOR_TOTAL}, name="primary family"))

In [ ]:

# =====================================================================
# Load the secondary family (interaction level, fixed 36-pair set)
# =====================================================================
SECONDARY_PATH = resolve_secondary_path()
secondary_raw, SECONDARY_PATH = read_table_any(SECONDARY_PATH, "secondary fixed-pair-set scores")
sec_patient_col = resolve_column(secondary_raw, PATIENT_ALIASES, "patient")
sec_id_col = resolve_column(secondary_raw, INTERACTION_ALIASES, "id_cp_interaction")
sec_score_col = resolve_column(secondary_raw, SCORE_ALIASES, "score")
sec_sender_col = resolve_column(secondary_raw, SENDER_ALIASES, "sender", required=False)
sec_receiver_col = resolve_column(secondary_raw, RECEIVER_ALIASES, "receiver", required=False)
secondary_raw[sec_patient_col] = secondary_raw[sec_patient_col].astype(str).str.strip()

SECONDARY_PROVENANCE = {"path": str(SECONDARY_PATH), "sha256": _sha256(SECONDARY_PATH)}

if sec_sender_col is not None and sec_receiver_col is not None:
    # Table is still per directed pair: verify the pair set is FIXED, then average over it.
    pairs = (secondary_raw[[sec_sender_col, sec_receiver_col]].drop_duplicates()
             .sort_values([sec_sender_col, sec_receiver_col]).reset_index(drop=True))
    core_types = sorted(set(pairs[sec_sender_col]) | set(pairs[sec_receiver_col]))
    assert len(pairs) == EXPECTED_SECONDARY_PAIRS, (
        f"fixed pair set has {len(pairs)} directed pairs, expected {EXPECTED_SECONDARY_PAIRS}")
    assert len(core_types) * len(core_types) == EXPECTED_SECONDARY_PAIRS, core_types
    agg = (secondary_raw.groupby([sec_id_col, sec_patient_col])
           .agg(_sum=(sec_score_col, "sum"), _n=(sec_score_col, "count")).reset_index())
    complete = agg["_n"] == EXPECTED_SECONDARY_PAIRS
    agg["_mean"] = np.where(complete, agg["_sum"] / EXPECTED_SECONDARY_PAIRS, np.nan)
    secondary_long = agg.rename(columns={sec_id_col: "id_cp_interaction",
                                         sec_patient_col: "patient"})[
        ["id_cp_interaction", "patient", "_mean", "_n"]].rename(columns={"_mean": "score",
                                                                         "_n": "n_pairs"})
    SECONDARY_PROVENANCE.update({"form": "per_pair", "core_cell_types": core_types})
else:
    secondary_long = secondary_raw.rename(columns={sec_id_col: "id_cp_interaction",
                                                   sec_patient_col: "patient",
                                                   sec_score_col: "score"})
    npair_col = next((c for c in secondary_long.columns
                      if str(c).lower() in {"n_pairs", "npairs", "n_pair", "pairs_used"}), None)
    if npair_col is not None:
        secondary_long["n_pairs"] = secondary_long[npair_col].astype(float)
        secondary_long.loc[secondary_long["n_pairs"] != EXPECTED_SECONDARY_PAIRS, "score"] = np.nan
    else:
        secondary_long["n_pairs"] = np.where(secondary_long["score"].notna(),
                                             EXPECTED_SECONDARY_PAIRS, 0)
    SECONDARY_PROVENANCE.update({"form": "pre_aggregated", "n_pairs_column": npair_col})

secondary_long = secondary_long[secondary_long["patient"].isin(PATIENTS)]
secondary_keys, V_SECONDARY = long_to_matrix(
    secondary_long, ["id_cp_interaction"], "patient", "score", PATIENTS)
A_SECONDARY = np.isfinite(V_SECONDARY)
secondary_keys["sender"] = "FIXED_36_PAIR_SET"
secondary_keys["receiver"] = "FIXED_36_PAIR_SET"
secondary_keys["interaction_name"] = secondary_keys["id_cp_interaction"].map(NAME_MAP).fillna("")

# Evaluability of the secondary family is a design expectation, recorded rather than enforced.
sec_eval_per_patient = A_SECONDARY.sum(axis=0)
sec_any = A_SECONDARY.any(axis=0)
SECONDARY_OBSERVED = {
    "total": int(sec_any.sum()),
    "good": int((sec_any & (GOOD_VEC > 0)).sum()),
    "poor": int((sec_any & (POOR_VEC > 0)).sum()),
}
SECONDARY_PROVENANCE["expected_evaluable"] = EXPECTED_SECONDARY_EVALUABLE
SECONDARY_PROVENANCE["observed_evaluable"] = SECONDARY_OBSERVED
SECONDARY_PROVENANCE["expectation_match"] = (
    SECONDARY_OBSERVED == EXPECTED_SECONDARY_EVALUABLE)
assert SECONDARY_OBSERVED["good"] >= MIN_EVAL_PER_GROUP, SECONDARY_OBSERVED
assert SECONDARY_OBSERVED["poor"] >= MIN_EVAL_PER_GROUP, SECONDARY_OBSERVED
LOG("secondary_loaded", n_endpoints=int(len(secondary_keys)), **SECONDARY_PROVENANCE)
display(pd.Series({**SECONDARY_OBSERVED, "endpoints": len(secondary_keys),
                   "expectation_match": SECONDARY_PROVENANCE["expectation_match"]},
                  name="secondary family evaluability"))


## The family pipeline

`evaluate_family` runs the complete prespecified procedure for one endpoint family and is called
twice, once for `primary` and once for `secondary`. The two families never share a BH correction,
never share a permutation subset, and neither is described as validating the other.

In [ ]:

# =====================================================================
# Evaluability, effect sizes, hybrid p-values
# =====================================================================
def describe_groups(V, good_vec, poor_vec):
    """Per-endpoint counts, medians and IQRs for each outcome group."""
    gmask = good_vec.astype(bool)
    pmask = poor_vec.astype(bool)
    Vg = np.where(gmask[None, :], V, np.nan)
    Vp = np.where(pmask[None, :], V, np.nan)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)  # all-NaN rows are expected
        out = {
            "median_good": np.nanmedian(Vg, axis=1),
            "median_poor": np.nanmedian(Vp, axis=1),
            "q25_good": np.nanpercentile(Vg, 25, axis=1),
            "q75_good": np.nanpercentile(Vg, 75, axis=1),
            "q25_poor": np.nanpercentile(Vp, 25, axis=1),
            "q75_poor": np.nanpercentile(Vp, 75, axis=1),
        }
    out["iqr_good"] = out["q75_good"] - out["q25_good"]
    out["iqr_poor"] = out["q75_poor"] - out["q25_poor"]
    out["median_difference"] = out["median_good"] - out["median_poor"]
    return out


def hybrid_pvalues(V, A, good_vec, sw, tested):
    """Exact where the endpoint is tie-free, tie-corrected asymptotic otherwise."""
    k = V.shape[0]
    p = np.where(tested, sw["p_asymptotic"], np.nan)
    method = np.array([""] * k, dtype=object)
    method[tested] = "asymptotic"
    tie_free = (sw["tie_term"] == 0) & tested
    gmask = good_vec.astype(bool)
    cache = {}
    for e in np.flatnonzero(tie_free):
        key = (int(sw["n_good"][e]), int(sw["n_poor"][e]), float(sw["u_good"][e]))
        if key not in cache:
            gv = V[e][A[e] & gmask]
            pv = V[e][A[e] & ~gmask]
            u_, p_, m_ = as_mw_result(
                _try_calls(mannwhitney_hybrid, (gv, pv), (gv, pv, "two-sided")))
            assert abs(u_ - sw["u_good"][e]) < 1e-9, (
                f"utils.mannwhitney_hybrid U={u_} disagrees with the sweep U={sw['u_good'][e]}")
            assert m_ == "exact", (
                "utils.mannwhitney_hybrid did not select the exact method for tie-free data; "
                f"it returned {m_!r}. The hybrid contract has diverged.")
            cache[key] = p_
        p[e] = cache[key]
        method[e] = "exact"
    return p, method, int(tie_free.sum()), len(cache)


def audit_asymptotic_against_utils(V, A, good_vec, sw, tested, p_hybrid, method, rng, n=None):
    """Assert the vectorised asymptotic sweep reproduces utils on real endpoints."""
    n = AUDIT_SAMPLE_N if n is None else n
    pool = np.flatnonzero(tested & (method == "asymptotic"))
    if pool.size == 0:
        return {"audited": 0, "max_p_error": 0.0, "max_rb_error": 0.0}
    sel = rng.choice(pool, size=min(n, pool.size), replace=False)
    gmask = good_vec.astype(bool)
    max_dp = max_drb = 0.0
    for e in sel:
        gv = V[e][A[e] & gmask]
        pv = V[e][A[e] & ~gmask]
        u_, p_, m_ = as_mw_result(
            _try_calls(mannwhitney_hybrid, (gv, pv), (gv, pv, "two-sided")))
        assert m_ == "asymptotic", (
            f"utils.mannwhitney_hybrid used {m_!r} on a tied endpoint; exact is invalid with ties")
        assert abs(u_ - sw["u_good"][e]) < 1e-9
        max_dp = max(max_dp, abs(p_ - p_hybrid[e]))
        rb_u = as_scalar(_try_calls(
            rank_biserial, (u_, gv.size, pv.size), (gv, pv)))
        max_drb = max(max_drb, abs(rb_u - rank_biserial_from_u(
            sw["u_good"][e], sw["n_good"][e], sw["n_poor"][e])))
    assert max_dp < 1e-9, (
        f"vectorised asymptotic p differs from utils.mannwhitney_hybrid by {max_dp:.3g}")
    assert max_drb < 1e-9, (
        f"vectorised rank-biserial differs from utils.rank_biserial by {max_drb:.3g}; check the "
        "Good-minus-Poor sign convention")
    return {"audited": int(sel.size), "max_p_error": float(max_dp), "max_rb_error": float(max_drb)}

In [ ]:

# =====================================================================
# Stratified patient bootstrap (vectorised, chunked)
# =====================================================================
def bootstrap_effects(V, A, good_vec, poor_vec, tested, n_resamples, seed):
    """95% percentile intervals for rank-biserial and median difference.

    Patients are resampled within outcome group. Cells are never resampled.
    """
    from collections import defaultdict

    k = V.shape[0]
    out = {nm: np.full(k, np.nan) for nm in
           ("rank_biserial_ci_low", "rank_biserial_ci_high",
            "median_difference_ci_low", "median_difference_ci_high")}
    rows = np.flatnonzero(tested)
    if rows.size == 0:
        return out, {"patterns": 0, "resamples": n_resamples}

    packed = np.packbits(A[rows], axis=1)
    groups = defaultdict(list)
    for i, row in zip(rows, packed):
        groups[row.tobytes()].append(i)

    gmask = good_vec.astype(bool)
    pmask = poor_vec.astype(bool)
    layout = {}
    sizes = set()
    for kb, idxs in groups.items():
        a = A[idxs[0]]
        gc = np.flatnonzero(a & gmask)
        pc = np.flatnonzero(a & pmask)
        layout[kb] = (gc, pc)
        sizes.update((gc.size, pc.size))

    # ONE draw per distinct group size, sorted, from a single seeded generator
    draws = draw_bootstrap_indices(sizes, n_resamples, seed)

    order = sorted(groups, key=lambda kb: (layout[kb][0].size, layout[kb][1].size, kb))
    current = None
    K = None
    for kb in order:
        gc, pc = layout[kb]
        n1, n2 = gc.size, pc.size
        if (n1, n2) != current:
            cg, cp = draws[n1]["counts"], draws[n2]["counts"]
            K = np.ascontiguousarray(
                (cg[:, :, None] * cp[:, None, :]).reshape(n_resamples, n1 * n2).T)
            current = (n1, n2)
        idxs = np.asarray(groups[kb])
        glo, ghi = draws[n1]["med_lo"], draws[n1]["med_hi"]
        plo, phi = draws[n2]["med_lo"], draws[n2]["med_hi"]
        for s in range(0, idxs.size, BOOT_ENDPOINT_CHUNK):
            ch = idxs[s:s + BOOT_ENDPOINT_CHUNK]
            Vg = np.ascontiguousarray(V[np.ix_(ch, gc)])
            Vp = np.ascontiguousarray(V[np.ix_(ch, pc)])

            H = ((Vg[:, :, None] > Vp[:, None, :]).astype(np.float64)
                 + 0.5 * (Vg[:, :, None] == Vp[:, None, :])).reshape(ch.size, n1 * n2)
            rb_boot = 2.0 * (H @ K) / (n1 * n2) - 1.0
            lo, hi = np.percentile(rb_boot, [CI_LOW_PCT, CI_HIGH_PCT], axis=1)
            out["rank_biserial_ci_low"][ch] = lo
            out["rank_biserial_ci_high"][ch] = hi
            del H, rb_boot

            Vgs, Vps = np.sort(Vg, axis=1), np.sort(Vp, axis=1)
            md_boot = (0.5 * (Vgs[:, glo] + Vgs[:, ghi])
                       - 0.5 * (Vps[:, plo] + Vps[:, phi]))
            lo, hi = np.percentile(md_boot, [CI_LOW_PCT, CI_HIGH_PCT], axis=1)
            out["median_difference_ci_low"][ch] = lo
            out["median_difference_ci_high"][ch] = hi
            del md_boot

    return out, {"patterns": len(groups), "resamples": n_resamples,
                 "group_sizes_drawn": sorted(int(s) for s in sizes)}


def audit_bootstrap_against_utils(V, A, good_vec, poor_vec, tested, boot, rng, n=25):
    """Record whether utils.bootstrap_ci reproduces the vectorised percentile intervals."""
    rows = np.flatnonzero(tested)
    if rows.size == 0:
        return {"status": "no_tested_endpoints"}
    sel = rng.choice(rows, size=min(n, rows.size), replace=False)
    gmask, pmask = good_vec.astype(bool), poor_vec.astype(bool)
    diffs = []
    try:
        for e in sel:
            gv, pv = V[e][A[e] & gmask], V[e][A[e] & pmask]
            res = _try_calls(
                bootstrap_ci,
                (V[e][None, :], np.flatnonzero(A[e] & gmask),
                 np.flatnonzero(A[e] & pmask), BOOTSTRAP_RESAMPLES, ANALYSIS_SEED),
                (gv, pv, rank_biserial),
                (gv, pv),
                ci=CI_LEVEL,
            )
            if isinstance(res, dict) and "rank_biserial_ci_low" in res:
                lo = float(np.ravel(res["rank_biserial_ci_low"])[0])
                hi = float(np.ravel(res["rank_biserial_ci_high"])[0])
            else:
                lo, hi = as_interval(res)
            diffs.append((abs(lo - boot["rank_biserial_ci_low"][e]),
                          abs(hi - boot["rank_biserial_ci_high"][e])))
    except (TypeError, AssertionError, ValueError) as exc:
        return {"status": "utils.bootstrap_ci signature not matched", "detail": str(exc)[:200]}
    arr = np.asarray(diffs)
    return {"status": "compared", "n": int(len(diffs)),
            "max_abs_difference": float(arr.max()) if arr.size else 0.0,
            "note": ("independent Monte-Carlo streams; small differences are resampling noise, "
                     "not disagreement about the statistic")}

In [ ]:

# =====================================================================
# Availability testing (separate BH family) and the permutation check
# =====================================================================
def availability_tests(A, good_vec, poor_vec):
    """Fisher exact per endpoint on availability x outcome, BH within its own family."""
    Af = A.astype(np.float64)
    a_good = (Af @ good_vec).astype(int)
    a_poor = (Af @ poor_vec).astype(int)
    n_g, n_p = int(good_vec.sum()), int(poor_vec.sum())
    cache = {}
    odds = np.empty(A.shape[0], dtype=float)
    pvals = np.empty(A.shape[0], dtype=float)
    for i in range(A.shape[0]):
        key = (a_good[i], a_poor[i])
        if key not in cache:
            cache[key] = fisher_2x2(key[0], n_g, key[1], n_p)
        odds[i], pvals[i] = cache[key]
    q = bh_qvalues(pvals)
    return {"availability_good": a_good, "availability_poor": a_poor,
            "availability_odds_ratio": odds, "availability_p": pvals,
            "availability_q": q, "availability_family_size": int(A.shape[0]),
            "distinct_tables": len(cache)}


def permutation_pvalue(n_extreme, n_valid):
    """(extreme + 1) / (valid + 1), taken from the shared API where its signature allows."""
    reference = (float(n_extreme) + 1.0) / (float(n_valid) + 1.0)
    try:
        val = as_scalar(_try_calls(permutation_p, (int(n_extreme), int(n_valid))),
                        ("p", "pvalue", "p_value"))
        if abs(val - reference) < 1e-12:
            BACKENDS.setdefault("permutation_p", "utils.permutation_p")
            return val
        BACKENDS["permutation_p"] = (
            "prespecified formula (utils.permutation_p returned a different convention)")
    except (TypeError, AssertionError, ValueError) as exc:
        BACKENDS["permutation_p"] = f"prespecified formula (utils.permutation_p unusable: {exc})"
    return reference


def permutation_check(V, A, sw, subset_rows, good_vec, n_perm, seed):
    """Patient-label permutation sensitivity check on the prespecified subset.

    The test statistic is the tie-corrected standardised |z| of the Mann-Whitney U. Within a
    fixed pair of group sizes it is a strictly monotone transform of the two-sided p-value, and
    it remains comparable when structural missingness makes the evaluable group sizes vary from
    one permutation to the next. Permuting labels changes neither the scores nor which patients
    are evaluable, so ranks and tie terms are computed once and reused.
    """
    k = A.shape[0]
    perm_p = np.full(k, np.nan)
    perm_valid = np.full(k, np.nan)
    if subset_rows.size == 0:
        return perm_p, perm_valid, {"subset": 0}

    Asub = A[subset_rows].astype(np.float64)
    Rsub = sw["R"][subset_rows] * Asub
    tt = sw["tie_term"][subset_rows][:, None]
    z_obs = np.abs(sw["z"][subset_rows])
    assert np.all(np.isfinite(z_obs)), "non-finite observed z in the permutation subset"

    n_pat = A.shape[1]
    n_good_total = int(good_vec.sum())
    rng = np.random.default_rng(seed)
    extreme = np.zeros(subset_rows.size, dtype=np.int64)
    valid = np.zeros(subset_rows.size, dtype=np.int64)

    done = 0
    while done < n_perm:
        m = int(min(PERM_CHUNK, n_perm - done))
        L = permutation_label_matrix(rng, n_pat, n_good_total, m)
        n1 = Asub @ L
        n2 = Asub @ (1.0 - L)
        U1 = Rsub @ L - n1 * (n1 + 1.0) / 2.0
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            _, z_perm, _ = mw_asymptotic_from_u(U1, n1, n2, tt)
        ok = (n1 >= MIN_EVAL_PER_GROUP) & (n2 >= MIN_EVAL_PER_GROUP) & np.isfinite(z_perm)
        valid += ok.sum(axis=1)
        extreme += (ok & (np.abs(z_perm) >= z_obs[:, None] - 1e-12)).sum(axis=1)
        done += m

    for j, e in enumerate(subset_rows):
        perm_p[e] = permutation_pvalue(int(extreme[j]), int(valid[j]))
        perm_valid[e] = int(valid[j])
    return perm_p, perm_valid, {
        "subset": int(subset_rows.size),
        "requested_permutations": int(n_perm),
        "min_valid": int(valid.min()), "max_valid": int(valid.max()),
        "statistic": "abs_tie_corrected_z",
        "resolution_floor": float(1.0 / (valid.min() + 1.0)),
    }

In [ ]:

# =====================================================================
# Minimum attainable p-value (diagnostic only - NOT a filter)
# =====================================================================
def min_attainable_p(sw, tested):
    """Lower bound on each endpoint's two-sided p under the most extreme ranking.

    Tie-free endpoints: the exact p at perfect separation, U = n_good * n_poor.
    Tied endpoints: the tie-corrected asymptotic p at U = n_good * n_poor. Perfect separation
    may be unreachable when a tie group straddles the two outcome groups, so this is a
    lower bound on the floor and the resulting "cannot attain" count is conservative.
    """
    k = len(tested)
    floor = np.full(k, np.nan)
    n1 = sw["n_good"].astype(float)
    n2 = sw["n_poor"].astype(float)
    tie_free = (sw["tie_term"] == 0) & tested
    tied = tested & ~tie_free

    if tied.any():
        p_t, _, _ = mw_asymptotic_from_u(n1[tied] * n2[tied], n1[tied], n2[tied],
                                         sw["tie_term"][tied])
        floor[tied] = p_t
    cache = {}
    for e in np.flatnonzero(tie_free):
        key = (int(n1[e]), int(n2[e]))
        if key not in cache:
            gv = np.arange(key[0], dtype=float) + 1000.0
            pv = np.arange(key[1], dtype=float)
            _, p_, m_ = as_mw_result(_try_calls(mannwhitney_hybrid, (gv, pv), (gv, pv, "two-sided")))
            assert m_ == "exact"
            cache[key] = p_
        floor[e] = cache[key]
    return floor

In [ ]:

# =====================================================================
# One family, end to end
# =====================================================================
def evaluate_family(V, A, keys, family, good_vec, poor_vec):
    keys = keys.reset_index(drop=True)
    k = V.shape[0]
    assert len(keys) == k

    n_good = (A.astype(float) @ good_vec).astype(int)
    n_poor = (A.astype(float) @ poor_vec).astype(int)
    nonzero = A & (np.nan_to_num(V, nan=0.0) != 0.0)
    nonzero_good = (nonzero.astype(float) @ good_vec).astype(int)
    nonzero_poor = (nonzero.astype(float) @ poor_vec).astype(int)
    nonzero_total = nonzero_good + nonzero_poor

    ok_group = (n_good >= MIN_EVAL_PER_GROUP) & (n_poor >= MIN_EVAL_PER_GROUP)
    ok_nonzero = nonzero_total >= MIN_NONZERO_TOTAL
    estimable = ok_group
    tested = ok_group & ok_nonzero
    exclusion_reason = np.select(
        [tested,
         (~ok_group) & (~ok_nonzero),
         ~ok_group,
         ~ok_nonzero],
        ["", "insufficient_evaluable_patients_and_nonzero_scores",
         "insufficient_evaluable_patients", "insufficient_nonzero_scores"],
        default="")

    desc = describe_groups(V, good_vec, poor_vec)
    sw = mw_sweep(V, A, good_vec, poor_vec)
    assert np.array_equal(sw["n_good"], n_good) and np.array_equal(sw["n_poor"], n_poor)

    p_value, p_method, n_tie_free, n_exact_calls = hybrid_pvalues(V, A, good_vec, sw, tested)
    rb = np.where(tested, rank_biserial_from_u(sw["u_good"], n_good, n_poor), np.nan)

    audit_rng = np.random.default_rng(ANALYSIS_SEED + 101)
    audit = audit_asymptotic_against_utils(V, A, good_vec, sw, tested, p_value, p_method, audit_rng)

    boot, boot_meta = bootstrap_effects(V, A, good_vec, poor_vec, tested,
                                        BOOTSTRAP_RESAMPLES, ANALYSIS_SEED)
    boot_audit = audit_bootstrap_against_utils(V, A, good_vec, poor_vec, tested, boot,
                                               np.random.default_rng(ANALYSIS_SEED + 202))

    avail = availability_tests(A, good_vec, poor_vec)

    # --- BH once, across exactly the tested endpoints of this family ------
    q_value = np.full(k, np.nan)
    tested_idx = np.flatnonzero(tested)
    bh_family_size = int(tested_idx.size)
    if bh_family_size:
        q_value[tested_idx] = bh_qvalues(p_value[tested_idx])

    # --- prespecified permutation subset ---------------------------------
    subset = np.zeros(k, dtype=bool)
    if bh_family_size:
        subset[tested_idx[q_value[tested_idx] < PERMUTATION_Q_GATE]] = True
        top = tested_idx[np.argsort(p_value[tested_idx], kind="stable")[:PERMUTATION_TOP_N]]
        subset[top] = True
    perm_p, perm_valid, perm_meta = permutation_check(
        V, A, sw, np.flatnonzero(subset), good_vec, PERMUTATION_RESAMPLES, ANALYSIS_SEED)

    floor = min_attainable_p(sw, tested)

    res = pd.DataFrame({
        "family": family,
        "id_cp_interaction": keys["id_cp_interaction"].to_numpy(),
        "sender": keys["sender"].to_numpy(),
        "receiver": keys["receiver"].to_numpy(),
        "interaction_name": keys["interaction_name"].to_numpy(),
        "n_good": n_good, "n_poor": n_poor,
        "missing_good": int(good_vec.sum()) - n_good,
        "missing_poor": int(poor_vec.sum()) - n_poor,
        "nonzero_good": nonzero_good, "nonzero_poor": nonzero_poor,
        "nonzero_total": nonzero_total,
        "median_good": desc["median_good"], "median_poor": desc["median_poor"],
        "median_difference": desc["median_difference"],
        "q25_good": desc["q25_good"], "q75_good": desc["q75_good"], "iqr_good": desc["iqr_good"],
        "q25_poor": desc["q25_poor"], "q75_poor": desc["q75_poor"], "iqr_poor": desc["iqr_poor"],
        "rank_biserial": rb,
        "rank_biserial_ci_low": boot["rank_biserial_ci_low"],
        "rank_biserial_ci_high": boot["rank_biserial_ci_high"],
        "median_difference_ci_low": boot["median_difference_ci_low"],
        "median_difference_ci_high": boot["median_difference_ci_high"],
        "mannwhitney_u": np.where(tested, sw["u_good"], np.nan),
        "mannwhitney_z": np.where(tested, sw["z"], np.nan),
        "tie_term": sw["tie_term"],
        "has_ties": sw["tie_term"] > 0,
        "p_value": p_value, "p_method": p_method, "q_value": q_value,
        "min_attainable_p": floor,
        "availability_good": avail["availability_good"],
        "availability_poor": avail["availability_poor"],
        "availability_odds_ratio": avail["availability_odds_ratio"],
        "availability_p": avail["availability_p"],
        "availability_q": avail["availability_q"],
        "availability_selection_flag": avail["availability_q"] < 0.05,
        "permutation_p": perm_p, "permutation_valid_n": perm_valid,
        "in_permutation_subset": subset,
        "estimable": estimable, "tested": tested, "exclusion_reason": exclusion_reason,
    })
    res["significant_q05"] = res["tested"] & (res["q_value"] < FDR_ALPHA)

    meta = {
        "family": family, "n_candidate": int(k), "n_estimable": int(estimable.sum()),
        "n_tested": bh_family_size, "bh_family_size": bh_family_size,
        "n_tie_free_tested": int(n_tie_free), "n_exact_pvalues_cached": int(n_exact_calls),
        "n_significant_q05": int(res["significant_q05"].sum()),
        "n_availability_flagged": int(res.loc[res["tested"], "availability_selection_flag"].sum()),
        "availability_family_size": avail["availability_family_size"],
        "availability_distinct_tables": avail["distinct_tables"],
        "mw_audit": audit, "bootstrap": boot_meta, "bootstrap_audit": boot_audit,
        "permutation": perm_meta,
    }
    return res, meta, sw

In [ ]:

# =====================================================================
# Run both families
# =====================================================================
RESULTS = {}
FAMILY_META = {}
SWEEPS = {}

RESULTS["primary"], FAMILY_META["primary"], SWEEPS["primary"] = evaluate_family(
    V_PRIMARY, A_PRIMARY, primary_keys, "primary", GOOD_VEC, POOR_VEC)
LOG("family_complete", **{k: v for k, v in FAMILY_META["primary"].items()
                          if not isinstance(v, dict)})

RESULTS["secondary"], FAMILY_META["secondary"], SWEEPS["secondary"] = evaluate_family(
    V_SECONDARY, A_SECONDARY, secondary_keys, "secondary", GOOD_VEC, POOR_VEC)
LOG("family_complete", **{k: v for k, v in FAMILY_META["secondary"].items()
                          if not isinstance(v, dict)})

FAMILY_SUMMARY = pd.DataFrame([
    {k: v for k, v in FAMILY_META[f].items() if not isinstance(v, dict)}
    for f in ("primary", "secondary")
])
display(FAMILY_SUMMARY)
display(pd.Series(BACKENDS, name="statistical backends used"))


## Result tables

`patient_level_primary_results.csv` is the primary family. The secondary family is written
separately, and a concatenated table carries the `family` column for downstream notebooks. Every
row of every table records `tested` and, where applicable, `exclusion_reason`, so the BH
denominator is reconstructable from the file itself.

In [ ]:

# =====================================================================
# Write result tables
# =====================================================================
COLUMN_ORDER = [
    "family", "id_cp_interaction", "sender", "receiver", "interaction_name",
    "n_good", "n_poor", "missing_good", "missing_poor",
    "nonzero_good", "nonzero_poor", "nonzero_total",
    "median_good", "median_poor", "median_difference",
    "q25_good", "q75_good", "iqr_good", "q25_poor", "q75_poor", "iqr_poor",
    "rank_biserial", "rank_biserial_ci_low", "rank_biserial_ci_high",
    "median_difference_ci_low", "median_difference_ci_high",
    "mannwhitney_u", "mannwhitney_z", "tie_term", "has_ties",
    "p_value", "p_method", "q_value", "min_attainable_p",
    "availability_good", "availability_poor", "availability_odds_ratio",
    "availability_p", "availability_q", "availability_selection_flag",
    "permutation_p", "permutation_valid_n", "in_permutation_subset",
    "estimable", "tested", "exclusion_reason", "significant_q05",
]
for fam in RESULTS:
    missing = [c for c in COLUMN_ORDER if c not in RESULTS[fam].columns]
    assert not missing, missing
    RESULTS[fam] = RESULTS[fam][COLUMN_ORDER]

write_table(RESULTS["primary"], PRIMARY_DIR / "patient_level_primary_results.csv")
write_table(RESULTS["secondary"], PRIMARY_DIR / "patient_level_secondary_results.csv")
write_table(pd.concat([RESULTS["primary"], RESULTS["secondary"]], ignore_index=True),
            PRIMARY_DIR / "patient_level_results_all_families.csv")
LOG("results_written", files=sorted(WRITTEN_FILES))


## Null / alternative diagnostic report

Everything below is descriptive. Ranked tables of non-significant endpoints are labelled as such
in the file itself, in a column that travels with the data, so a downstream reader cannot mistake
an exploratory ranking for a finding. Rendering of these distributions is deferred to
`07_figures_and_tables.ipynb`, which owns all plotting; here they are emitted as machine-readable
tables.

In [ ]:

# =====================================================================
# Diagnostics
# =====================================================================
NOT_SIG_LABEL = "NOT SIGNIFICANT AT q<0.05 - EXPLORATORY RANKING ONLY, NOT A FINDING"
PVAL_BINS = np.linspace(0.0, 1.0, 21)
Q_THRESHOLDS = [0.001, 0.01, 0.05, 0.10, 0.20, 0.50, 1.0]
QUANTILES = [0.0, 0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99, 1.0]

DIAGNOSTICS = {}


def family_diagnostics(res, meta, fam):
    t = res[res["tested"]]
    counts = pd.DataFrame([{
        "family": fam,
        "n_candidate_endpoints": meta["n_candidate"],
        "n_estimable_endpoints": meta["n_estimable"],
        "n_tested_endpoints": meta["n_tested"],
        "bh_family_size": meta["bh_family_size"],
        "n_excluded_group_size": int((res["exclusion_reason"]
                                      == "insufficient_evaluable_patients").sum()),
        "n_excluded_nonzero": int((res["exclusion_reason"]
                                   == "insufficient_nonzero_scores").sum()),
        "n_excluded_both": int((res["exclusion_reason"]
                                == "insufficient_evaluable_patients_and_nonzero_scores").sum()),
        "n_significant_q05": int(t["significant_q05"].sum()),
        "n_q_below_0.10": int((t["q_value"] < 0.10).sum()),
        "n_availability_flagged": meta["n_availability_flagged"],
        "n_exact_pmethod": int((t["p_method"] == "exact").sum()),
        "n_asymptotic_pmethod": int((t["p_method"] == "asymptotic").sum()),
    }])
    write_table(counts, DIAG_DIR / f"counts_{fam}.csv")

    hist = np.histogram(t["p_value"].to_numpy(), bins=PVAL_BINS)[0]
    ph = pd.DataFrame({"family": fam, "bin_low": PVAL_BINS[:-1], "bin_high": PVAL_BINS[1:],
                       "count": hist})
    ph["expected_under_global_null"] = len(t) / (len(PVAL_BINS) - 1)
    write_table(ph, DIAG_DIR / f"pvalue_histogram_{fam}.csv")

    qd = pd.DataFrame({"family": fam, "threshold": Q_THRESHOLDS,
                       "n_endpoints_at_or_below":
                           [int((t["q_value"] <= th).sum()) for th in Q_THRESHOLDS]})
    write_table(qd, DIAG_DIR / f"qvalue_distribution_{fam}.csv")

    ties = pd.DataFrame([{
        "family": fam,
        "n_tested": len(t),
        "n_with_ties": int(t["has_ties"].sum()),
        "fraction_with_ties": float(t["has_ties"].mean()) if len(t) else np.nan,
        "n_tie_free": int((~t["has_ties"]).sum()),
        "median_tie_term": float(t["tie_term"].median()) if len(t) else np.nan,
        "max_tie_term": float(t["tie_term"].max()) if len(t) else np.nan,
    }])
    write_table(ties, DIAG_DIR / f"tie_frequency_{fam}.csv")

    gs = (t.groupby(["n_good", "n_poor"]).size().rename("n_endpoints").reset_index())
    gs.insert(0, "family", fam)
    write_table(gs, DIAG_DIR / f"group_size_distribution_{fam}.csv")

    eff = t[["rank_biserial", "median_difference", "p_value", "q_value"]].quantile(QUANTILES)
    eff.index.name = "quantile"
    eff = eff.reset_index()
    eff.insert(0, "family", fam)
    write_table(eff, DIAG_DIR / f"effect_size_distribution_{fam}.csv")

    sig = t[t["significant_q05"]].sort_values("q_value")
    write_table(sig, DIAG_DIR / f"significant_q05_{fam}.csv")

    nonsig = t[~t["significant_q05"]].sort_values(["q_value", "p_value"]).head(50).copy()
    nonsig.insert(0, "significance_label", NOT_SIG_LABEL)
    write_table(nonsig, DIAG_DIR / f"top_ranked_NOT_significant_{fam}.csv")

    DIAGNOSTICS[fam] = {"counts": counts, "pvalue_histogram": ph, "qvalues": qd,
                        "ties": ties, "group_sizes": gs, "effects": eff,
                        "n_significant": int(len(sig))}
    return counts


for fam in ("primary", "secondary"):
    display(family_diagnostics(RESULTS[fam], FAMILY_META[fam], fam))

for fam in ("primary", "secondary"):
    n_sig = DIAGNOSTICS[fam]["n_significant"]
    if n_sig == 0:
        display(pd.Series(
            {"family": fam,
             "endpoints_surviving_FDR_at_q<0.05": 0,
             "statement": ("Zero endpoints are associated with clinical-outcome group at "
                           "Benjamini-Hochberg q<0.05 in this family. No nominal p-value is "
                           "promoted, no threshold is changed, and no non-significant result "
                           "is relabelled.")},
            name=f"{fam} FDR outcome"))


### Attainability of the FDR threshold (quantified, deliberately not acted upon)

The `>= 6` / `>= 6` evaluability rule admits endpoints that cannot reach `q < 0.05` under any
data configuration. At 6 Good versus 6 Poor the smallest attainable two-sided exact p-value is
2.164502e-03, so at BH rank 1 that endpoint could only be rejected in a family of at most
`floor(0.05 / 2.164502e-03) = 23` endpoints. Such endpoints still add 1 each to the BH
denominator and therefore raise the q-value of every other endpoint in the family.

The table below quantifies how much of each family is in that position. A Tarone-style
attainability filter would remove them from the denominator, and is **deliberately out of scope
for this run**: nothing here changes the tested family, the denominator, or any reported q-value.
The numbers exist so the decision can be revisited with evidence.

The bound used is the conservative rank-1 criterion (`min_attainable_p <= alpha / m`). An
endpoint failing it could in principle still be rejected at a higher BH rank if many other
endpoints were simultaneously rejected, so this count is an upper bound on the endpoints that are
*definitely* unrejectable at rank 1, not a proof of unrejectability.

In [ ]:

# =====================================================================
# Attainability diagnostic (reported only; no filtering is applied)
# =====================================================================
from scipy.stats import mannwhitneyu as _mwu

assert abs(_mwu(np.arange(6) + 10.0, np.arange(6.0), alternative="two-sided",
                method="exact").pvalue - 0.0021645021645021645) < 1e-15
assert int(np.floor(FDR_ALPHA / 0.0021645021645021645)) == 23

attain_rows = []
for fam in ("primary", "secondary"):
    res = RESULTS[fam]
    t = res[res["tested"]].copy()
    m = len(t)
    if m == 0:
        continue
    t["max_supportable_family_size"] = np.floor(FDR_ALPHA / t["min_attainable_p"])
    t["can_reach_q05_at_rank1"] = t["min_attainable_p"] <= FDR_ALPHA / m
    attain_rows.append({
        "family": fam,
        "bh_family_size": m,
        "alpha_over_m": FDR_ALPHA / m,
        "n_can_reach_q05_at_rank1": int(t["can_reach_q05_at_rank1"].sum()),
        "n_cannot_reach_q05_at_rank1": int((~t["can_reach_q05_at_rank1"]).sum()),
        "fraction_cannot_reach": float((~t["can_reach_q05_at_rank1"]).mean()),
        "n_at_6v6": int(((t["n_good"] == 6) & (t["n_poor"] == 6)).sum()),
        "n_tied_endpoints_unattainable": int((~t["can_reach_q05_at_rank1"] & t["has_ties"]).sum()),
        "median_min_attainable_p": float(t["min_attainable_p"].median()),
        "hypothetical_denominator_if_filtered": int(t["can_reach_q05_at_rank1"].sum()),
        "action_taken": "none - reported only, no endpoint was removed from any BH family",
    })
    detail = (t.groupby(["n_good", "n_poor", "has_ties"])
              .agg(n_endpoints=("min_attainable_p", "size"),
                   min_attainable_p=("min_attainable_p", "min"),
                   max_attainable_p=("min_attainable_p", "max"),
                   n_cannot_reach=("can_reach_q05_at_rank1", lambda s: int((~s).sum())))
              .reset_index())
    detail.insert(0, "family", fam)
    write_table(detail, DIAG_DIR / f"attainability_by_group_size_{fam}.csv")

ATTAINABILITY = pd.DataFrame(attain_rows)
write_table(ATTAINABILITY, DIAG_DIR / "attainability_summary.csv")
display(ATTAINABILITY)
LOG("attainability_reported", rows=ATTAINABILITY.to_dict(orient="records"))


## Verification

The notebook fails loudly rather than reporting an unverified result. Diagnostic files for each
check are written **before** the corresponding assertion, so a failure is inspectable.

One requested check is weakened deliberately and the reason is stated here rather than buried:
the sign of the rank-biserial correlation and the sign of the median difference are **not**
mathematically required to agree. They are different functionals - one measures stochastic
dominance over all patient pairs, the other compares two location estimates - and they can
legitimately disagree on tie-free data (for example when one group is shifted low but has a
heavy upper tail). What *is* guaranteed, and is asserted here, is that perfect dominance
(`|rank_biserial| = 1`) forces the median difference to share its sign. Every remaining sign
disagreement is written to `sign_disagreements_<family>.csv` with its tie status, and the count
is reported rather than suppressed.

In [ ]:

# =====================================================================
# Verification assertions
# =====================================================================
VERIFICATION = {}

for fam in ("primary", "secondary"):
    res = RESULTS[fam]
    meta = FAMILY_META[fam]
    t = res[res["tested"]]

    # 1. BH denominator equals this family's own tested count, exactly.
    assert meta["bh_family_size"] == int(res["tested"].sum()), fam
    assert int(t["q_value"].notna().sum()) == meta["bh_family_size"], fam
    assert int(res.loc[~res["tested"], "q_value"].notna().sum()) == 0, (
        f"{fam}: an untested endpoint carries a q-value")
    if meta["bh_family_size"]:
        m = meta["bh_family_size"]
        p_sorted = np.sort(t["p_value"].to_numpy())
        q_recomputed = np.minimum.accumulate(
            (m * p_sorted / np.arange(1, m + 1))[::-1])[::-1].clip(0, 1)
        assert np.allclose(np.sort(t["q_value"].to_numpy()), q_recomputed, atol=1e-12), (
            f"{fam}: q-values are not BH over exactly the tested family")

    # 2. exact p-values only where the endpoint is genuinely tie-free
    assert not (t["p_method"] == "exact").any() or (~t.loc[t["p_method"] == "exact",
                                                           "has_ties"]).all(), fam
    assert (t.loc[t["has_ties"], "p_method"] == "asymptotic").all(), (
        f"{fam}: a tied endpoint was given an exact p-value")
    assert set(t["p_method"]) <= {"exact", "asymptotic"}, set(t["p_method"])

    # 3. effect-size sign versus median difference
    disagree = t[(np.sign(t["rank_biserial"]) * np.sign(t["median_difference"])) < 0].copy()
    disagree["explanation"] = np.where(
        disagree["has_ties"], "tied ranking",
        "location versus dominance divergence (documented, not an error)")
    write_table(disagree, DIAG_DIR / f"sign_disagreements_{fam}.csv")
    perfect = t[np.abs(t["rank_biserial"]) == 1.0]
    assert np.all(np.sign(perfect["median_difference"]) == np.sign(perfect["rank_biserial"])), (
        f"{fam}: perfect dominance disagrees with the median difference - this is impossible "
        "and indicates a sign-convention error")

    # 4. every bootstrap CI contains its point estimate
    tol = 1e-9
    bad = t[((t["rank_biserial"] < t["rank_biserial_ci_low"] - tol)
             | (t["rank_biserial"] > t["rank_biserial_ci_high"] + tol)
             | (t["median_difference"] < t["median_difference_ci_low"] - tol)
             | (t["median_difference"] > t["median_difference_ci_high"] + tol))]
    write_table(bad, DIAG_DIR / f"bootstrap_ci_violations_{fam}.csv")
    assert len(bad) == 0, (
        f"{fam}: {len(bad)} bootstrap percentile intervals exclude their own point estimate; "
        f"see {DIAG_DIR / f'bootstrap_ci_violations_{fam}.csv'}")
    assert t[["rank_biserial_ci_low", "rank_biserial_ci_high",
              "median_difference_ci_low", "median_difference_ci_high"]].notna().all().all(), fam
    assert (t["rank_biserial_ci_low"] <= t["rank_biserial_ci_high"] + tol).all(), fam
    assert (t["median_difference_ci_low"] <= t["median_difference_ci_high"] + tol).all(), fam

    # 5. permutation p-values respect their own resolution floor
    perm = res[res["in_permutation_subset"]]
    if len(perm):
        assert perm["permutation_valid_n"].notna().all() and perm["permutation_p"].notna().all()
        assert np.all(perm["permutation_p"] >= 1.0 / (perm["permutation_valid_n"] + 1.0) - 1e-15)
        assert np.all(perm["permutation_p"] <= 1.0)
        assert np.all(perm["permutation_valid_n"] <= PERMUTATION_RESAMPLES)
        assert bool(perm["tested"].all()), f"{fam}: an untested endpoint entered the permutation subset"
    assert int(res.loc[~res["in_permutation_subset"], "permutation_p"].notna().sum()) == 0, fam

    # 6. evaluability rules were applied as prespecified
    assert (t["n_good"] >= MIN_EVAL_PER_GROUP).all() and (t["n_poor"] >= MIN_EVAL_PER_GROUP).all()
    assert (t["nonzero_total"] >= MIN_NONZERO_TOTAL).all()
    assert (res.loc[~res["tested"], "exclusion_reason"] != "").all(), (
        f"{fam}: an excluded endpoint carries no exclusion_reason")
    assert (res.loc[res["tested"], "exclusion_reason"] == "").all()
    assert (t["n_good"] + t["missing_good"] == N_GOOD_TOTAL).all()
    assert (t["n_poor"] + t["missing_poor"] == N_POOR_TOTAL).all()

    # 7. availability family is corrected separately and completely
    assert int(res["availability_q"].notna().sum()) == len(res), fam
    assert meta["availability_family_size"] == len(res), fam

    VERIFICATION[fam] = {
        "bh_denominator": meta["bh_family_size"],
        "sign_disagreements": int(len(disagree)),
        "bootstrap_ci_violations": int(len(bad)),
        "permutation_subset": int(len(perm)),
        "n_significant_q05": int(t["significant_q05"].sum()),
    }

# families are independent: no shared correction, no cross-family filtering
assert set(RESULTS["primary"]["family"]) == {"primary"}
assert set(RESULTS["secondary"]["family"]) == {"secondary"}
assert FAMILY_META["primary"]["bh_family_size"] != 0 or FAMILY_META["secondary"]["bh_family_size"] != 0

display(pd.DataFrame(VERIFICATION).T)
LOG("verification_passed", **{f: VERIFICATION[f] for f in VERIFICATION})

In [ ]:

# =====================================================================
# Machine-readable success record
# =====================================================================
SUCCESS = {
    "notebook": NOTEBOOK_ID,
    "status": "success",
    "run_started": RUN_STARTED,
    "run_finished": dt.datetime.now().isoformat(timespec="seconds"),
    "config_path": str(CONFIG_PATH),
    "config_sha256": CONFIG_SHA256,
    "environment": ENVIRONMENT,
    "analysis_seed": ANALYSIS_SEED,
    "bootstrap_resamples": BOOTSTRAP_RESAMPLES,
    "permutation_resamples": PERMUTATION_RESAMPLES,
    "fdr_method": FDR_METHOD,
    "fdr_alpha": FDR_ALPHA,
    "outcome_column": OUTCOME_COLUMN,
    "n_patients": len(PATIENTS),
    "n_good": N_GOOD_TOTAL,
    "n_poor": N_POOR_TOTAL,
    "patients": list(PATIENTS),
    "secondary_provenance": SECONDARY_PROVENANCE,
    "statistical_backends": BACKENDS,
    "kernel_selftest": {k: float(v) for k, v in SELFTEST.items()},
    "families": {f: FAMILY_META[f] for f in FAMILY_META},
    "verification": VERIFICATION,
    "attainability": ATTAINABILITY.to_dict(orient="records"),
    "written_files": WRITTEN_FILES,
    "claims": {
        "unit_of_analysis": "patient",
        "families_are_independent": True,
        "secondary_validates_primary": False,
        "pooled_pvalues_used_as_patient_evidence": False,
        "permutation_is_sensitivity_check_only": True,
        "attainability_filter_applied": False,
        "fdr_threshold_modified": False,
    },
}
SUCCESS_PATH = guarded_path(PRIMARY_DIR / "_04_patient_level_inference_success.json")
SUCCESS_PATH.write_text(json.dumps(SUCCESS, indent=2, default=str) + "\n")
LOG("success", success_path=str(SUCCESS_PATH),
    n_significant={f: VERIFICATION[f]["n_significant_q05"] for f in VERIFICATION})
assert SUCCESS_PATH.is_file()
display(pd.Series({"success record": str(SUCCESS_PATH),
                   "primary tested endpoints": FAMILY_META["primary"]["bh_family_size"],
                   "primary q<0.05": VERIFICATION["primary"]["n_significant_q05"],
                   "secondary tested endpoints": FAMILY_META["secondary"]["bh_family_size"],
                   "secondary q<0.05": VERIFICATION["secondary"]["n_significant_q05"]},
                  name="04_patient_level_inference"))